# Day 2 - AutoML로 콘크리트 압축강도 예측·해석·개선하기

## 📌 Day 1 복습

Day 1에서는 **베어링 센서 신호**를 이용해 "정상 / 고장(고장 종류)"을 구분하는 **분류(Classification)** 문제를 다뤘습니다.
- 입력: 진동 신호 (시계열 → FFT/주파수 특징)
- 출력: 4개 클래스 중 하나 (Normal, Inner Race, Outer Race, Ball)
- 모델: 1D-CNN, CNN-LSTM (딥러닝)

## 🎯 Day 2의 목표 : 회귀(Regression) 문제로 전환

오늘은 조금 다른 종류의 문제를 다룹니다.

> **"이 레시피(원료 비율)로 콘크리트를 만들면, 28일 후 압축강도가 몇 MPa이 나올까?"**

이번엔 클래스를 맞추는 게 아니라, **숫자(연속값)를 예측**하는 문제입니다. 이런 문제를 **회귀(Regression)**라고 부릅니다.

또한 이번엔 딥러닝 모델을 직접 코딩하는 대신, **AutoML 도구(PyCaret)**를 사용합니다.

## ✅ 오늘 다룰 내용

1. 콘크리트 압축강도 데이터셋을 살펴보고, 변수들의 의미를 이해한다
2. PyCaret으로 여러 머신러닝 모델을 한 번에 비교한다 (`setup()` + `compare_models()`)
3. `interpret_model()`로 모델이 **왜** 그렇게 예측했는지 해석한다 (SHAP)
4. `tune_model()`로 성능을 추가로 개선한다

## STEP 1. 데이터셋 소개

오늘 사용할 데이터는 **UCI Concrete Compressive Strength** 데이터셋입니다.
콘크리트를 만들 때 섞는 **8가지 원료/조건**과, 그 결과로 나온 **28일 압축강도**가 기록되어 있습니다 (총 1030개 샘플).

### 입력 변수 (8개) - "레시피"

| 컬럼명 | 의미 | 단위 |
|---|---|---|
| `cement` | 시멘트 | kg/m³ |
| `blast_furnace_slag` | 고로 슬래그 (시멘트 대체 재료) | kg/m³ |
| `fly_ash` | 플라이애시 (석탄재, 혼합재) | kg/m³ |
| `water` | 물 | kg/m³ |
| `superplasticizer` | 고성능 감수제 (유동성 향상 첨가제) | kg/m³ |
| `coarse_aggregate` | 굵은 골재 (자갈) | kg/m³ |
| `fine_aggregate` | 잔골재 (모래) | kg/m³ |
| `age` | 양생 일수 | day |

### 출력 변수 (1개) - "결과"

| 컬럼명 | 의미 | 단위 |
|---|---|---|
| `concrete_compressive_strength` | 28일 압축강도 | MPa |

### 💡 현장 기준 참고

KS L 5201 (포틀랜드 시멘트 규격)에서는 **1종 시멘트의 28일 압축강도가 42.5 MPa 이상**이어야 합니다.
오늘 만드는 모델이 예측한 값이 이 기준을 기준으로 "쓸만한 정확도"인지 판단하는 데 사용해보겠습니다.


In [1]:
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (OS별 한글 폰트 자동 설정)
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv("concrete_data.csv")

# 컬럼명 앞뒤 공백 제거 (예: 'fine_aggregate ' -> 'fine_aggregate')
df.columns = df.columns.str.strip()

print(f"데이터 크기: {df.shape}")
df.head()

데이터 크기: (1030, 9)


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,concrete_compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


## STEP 4. 왜 AutoML인가?

회귀 문제를 풀 수 있는 머신러닝 모델은 정말 많습니다: 선형회귀, 랜덤포레스트, XGBoost, LightGBM, 신경망 등...

**어떤 모델이 우리 데이터에 가장 잘 맞을지는, 직접 다 돌려보기 전까지는 알기 어렵습니다.**

### 🍳 비유: AutoML = 여러 셰프에게 같은 재료를 주고 요리를 시켜보기

같은 재료(데이터)를 여러 셰프(모델)에게 주고 요리(학습)를 시킨 뒤, 누구 요리가 가장 맛있는지(예측이 정확한지) 비교하는 것과 비슷합니다.

**PyCaret**은 이 과정을 자동화해주는 라이브러리입니다.
- 데이터 전처리(결측치 처리, 정규화, train/test 분리)를 자동으로 해주고
- 수십 개의 머신러닝 모델을 한 번에 학습/평가해서 비교해줍니다

코드 몇 줄로 "이 데이터에 머신러닝이 통할까?"를 빠르게 검증할 수 있습니다.


## STEP 5. PyCaret 환경 설정 - `setup()`

PyCaret을 쓰기 위한 첫 단계는 `setup()` 함수 호출입니다. 이 한 줄이 다음을 자동으로 처리합니다:

- 데이터를 **학습용(train) / 검증용(test)**으로 분리
- 결측치 처리, 수치형 변수 정규화 등 전처리
- 각 컬럼의 데이터 타입(수치형/범주형) 자동 인식

실행하면 아래와 같은 **설정 요약 테이블**이 출력됩니다. 주요 항목 해석:

| 항목 | 의미 |
|---|---|
| Session id | 실험 재현을 위한 랜덤 시드 |
| Target | 예측하고자 하는 변수 (`concrete_compressive_strength`) |
| Target type | Regression (회귀) |
| Original data shape | 원본 데이터 크기 |
| Transformed data shape | 전처리 후 데이터 크기 |
| Numeric features | 수치형 입력 변수 개수 |

`session_id`는 매번 같은 결과가 나오도록 고정하는 랜덤 시드입니다 (실험 재현성을 위해 고정).


In [3]:
from pycaret.regression import *

s = setup(data=df, target='concrete_compressive_strength', session_id=123)

,Description,Value
0,Session id,123
1,Target,concrete_compressive_strength
2,Target type,Regression
3,Original data shape,"(1030, 9)"
4,Transformed data shape,"(1030, 9)"
5,Transformed train set shape,"(721, 9)"
6,Transformed test set shape,"(309, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


## STEP 6. 모델 비교 - `compare_models()`

`compare_models()`를 호출하면, PyCaret이 등록된 모든 회귀 모델(선형회귀, 랜덤포레스트, XGBoost, LightGBM 등)을 학습시키고,
**교차검증(cross-validation)** 결과를 성능 좋은 순서로 정렬한 표로 보여줍니다.

### 주요 평가지표

| 지표 | 의미 | 좋은 방향 |
|---|---|---|
| **R²** | 모델이 데이터의 분산을 얼마나 잘 설명하는지 (1에 가까울수록 좋음) | ↑ 클수록 좋음 |
| **RMSE** | 평균 오차 크기 (실제값과 같은 단위, MPa) - 큰 오차에 더 민감 | ↓ 작을수록 좋음 |
| **MAE** | 평균 절대 오차 (MPa) | ↓ 작을수록 좋음 |

### 💡 "쓸만한 모델인가?" 판단하기

예를 들어 RMSE가 4 MPa라면, 평균적으로 예측값이 실제값과 **±4 MPa** 정도 차이가 난다는 뜻입니다.
KS 기준이 42.5 MPa인 상황에서, 이 정도 오차가 "기준 통과 여부를 가를 정도로 큰 오차인지" 현장 맥락에서 판단해봐야 합니다.

(참고: 이 실행은 약 10초 정도 걸립니다.)


In [4]:
best_model = compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,3.2006,21.9523,4.6356,0.9215,0.1511,0.1106,0.1770
et,Extra Trees Regressor,3.2530,23.4571,4.7985,0.9166,0.1527,0.1128,0.0150
gbr,Gradient Boosting Regressor,3.7650,27.6610,5.2383,0.9007,0.1634,0.1295,0.0120
rf,Random Forest Regressor,3.6619,28.2038,5.2328,0.8993,0.1679,0.1290,0.0230
dt,Decision Tree Regressor,4.5467,43.9027,6.5348,0.8442,0.2165,0.1580,0.0030
ada,AdaBoost Regressor,6.1832,57.2779,7.5252,0.7944,0.2782,0.2605,0.0080
knn,K Neighbors Regressor,7.4127,94.3747,9.6708,0.6625,0.3208,0.2894,0.0070
ridge,Ridge Regression,8.3627,113.1336,10.6141,0.5892,0.3398,0.3216,0.0030
lar,Least Angle Regression,8.3627,113.1337,10.6141,0.5892,0.3398,0.3216,0.0030
lr,Linear Regression,8.3627,113.1337,10.6141,0.5892,0.3398,0.3216,0.2700


## STEP 7. 결과 해석하기

`compare_models()`가 출력한 표는 화면에 보여지는 것뿐, 코드로 다시 활용하려면 `pull()`로 표 자체를 DataFrame으로 가져올 수 있습니다.

또한 `best_model` 변수에는 **1위 모델 객체**가 자동으로 저장되어 있습니다.

### 모델을 선택하는 기준은 "1위"만이 아닙니다

- **정확도가 1위인 모델**이 항상 최선은 아닙니다.
- 학습/예측 속도(`TT (Sec)`)가 중요한 경우 → 약간 덜 정확해도 빠른 모델 선택
- 모델의 동작을 설명해야 하는 경우(해석성) → 복잡한 모델보다 단순한 모델 선택

다음 세션(세션 2)에서 이 "해석 가능성" 문제를 더 자세히 다룹니다.


In [5]:
results_df = pull()

print("1위 모델:", best_model)
print()
results_df[['Model', 'MAE', 'RMSE', 'R2', 'TT (Sec)']]

1위 모델: LGBMRegressor(n_jobs=-1, random_state=123)



,Model,MAE,RMSE,R2,TT (Sec)
lightgbm,Light Gradient Boosting Machine,3.2006,4.6356,0.9215,0.177
et,Extra Trees Regressor,3.2530,4.7985,0.9166,0.015
gbr,Gradient Boosting Regressor,3.7650,5.2383,0.9007,0.012
rf,Random Forest Regressor,3.6619,5.2328,0.8993,0.023
dt,Decision Tree Regressor,4.5467,6.5348,0.8442,0.003
ada,AdaBoost Regressor,6.1832,7.5252,0.7944,0.008
knn,K Neighbors Regressor,7.4127,9.6708,0.6625,0.007
ridge,Ridge Regression,8.3627,10.6141,0.5892,0.003
lar,Least Angle Regression,8.3627,10.6141,0.5892,0.003
lr,Linear Regression,8.3627,10.6141,0.5892,0.270


## STEP 8. 블랙박스 문제 - "모델이 왜 그렇게 예측했을까?"

1위 모델 **LightGBM**은 수백 개의 작은 의사결정나무를 조합해서 예측값을 만듭니다.
정확도는 높지만, 사람이 "왜 38 MPa로 예측했는지"를 직관적으로 설명하기 어렵습니다.

### 🏭 현장에서 문제가 되는 순간

> "이 배합으로 만들면 38 MPa라고 하는데... 왜요? 어떤 재료를 더 넣어야 강도가 올라가나요?"

담당자가 이 질문에 답하지 못하면, 아무리 정확한 모델이라도 **현장에서 신뢰받지 못하고 사용되지 않습니다.**

### 💡 PyCaret의 해답 - `interpret_model()`

PyCaret은 **SHAP(SHapley Additive exPlanations)**이라는 기법을 이용해,
"각 변수가 예측값에 얼마나, 어떤 방향으로 기여했는지"를 한 줄 코드로 시각화해줍니다.

### SHAP 그래프 읽는 법

- **y축**: 변수 이름 — 위에 있을수록 예측에 미치는 영향이 큰 변수
- **x축(SHAP value)**: 그 변수가 예측값을 얼마나, 어느 방향으로 바꿨는지
  - 오른쪽(양수) = 압축강도를 **높이는** 방향
  - 왼쪽(음수) = 압축강도를 **낮추는** 방향
- **점의 색**: 해당 변수의 실제 값 (빨강 = 값이 큼, 파랑 = 값이 작음)

(실행에 10~20초 정도 걸릴 수 있습니다.)

In [ ]:
interpret_model(best_model)

### 📊 SHAP 결과 해석

| 변수 | 그래프에서 보이는 패턴 | 현장 의미 |
|---|---|---|
| `age` (양생 기간) | 빨간 점(양생 오래함)이 오른쪽에 많음 | 오래 양생할수록 강도 ↑ |
| `cement` (시멘트량) | 빨간 점이 오른쪽에 많음 | 시멘트 많을수록 강도 ↑ |
| `water` (물) | 빨간 점(물 많음)이 왼쪽에 많음 | 물 많을수록 강도 ↓ |

### 🤔 토론해보기

- "양생 기간을 7일 더 늘리면 강도가 얼마나 달라질까?" — 이 그래프만으로 답할 수 있을까요, 아니면 추가 분석이 필요할까요?
- "물-시멘트 비율을 낮추면 강도에 어떤 효과가 있을까?" — `water`, `cement` 패턴이 이 질문에 어떤 근거를 주나요?
- 이 그래프를 **배합 설계 담당자**에게 보여준다면, 어떤 식으로 설명하는 게 좋을까요?

> 핵심: AI가 "정답"만 주는 게 아니라 **"왜 그런 정답이 나왔는지에 대한 근거"**까지 함께 제공한다는 점이 중요합니다.
> 이 근거가 있어야 현장에서 AI의 제안을 신뢰하고 실제 배합 설계에 참고할 수 있습니다.

## STEP 8-2. 샘플 하나 들여다보기 - SHAP Force Plot

Summary Plot은 "전체 데이터에서 평균적으로 어떤 변수가 중요한가"를 보여줬습니다.

**Force Plot**은 반대로 **샘플 하나**를 집어서 "이 배합이 구체적으로 왜 이 수치가 나왔는가"를 설명합니다.

```
기준값(전체 평균 예측) ≈ 35 MPa
    → cement 많음:  +8 MPa  (강도 높이는 방향 →)
    → water 많음:   -4 MPa  (강도 낮추는 방향 ←)
    → age 짧음:     -1 MPa
    ─────────────────────────
    최종 예측:       38 MPa
```

현장에서 "왜 이 배합은 기준 미달인가요?"라는 질문에 직접 답할 수 있는 그래프입니다.

`observation` 인자로 몇 번째 샘플을 분석할지 지정합니다 (0부터 시작).

In [ ]:
# 0번째 샘플 (첫 번째 배합)의 예측을 Force Plot으로 설명
print(f"분석할 샘플의 실제 압축강도: {df['concrete_compressive_strength'].iloc[0]} MPa")
interpret_model(best_model, plot='reason', observation=0)

## STEP 9. 특정 변수에 집중해서 해석하기

`interpret_model()`은 전체 변수의 요약 외에도, **특정 변수 하나**가 예측값에 미치는 영향을 더 자세히 볼 수 있습니다.

STEP 3에서 `age`(양생 기간)가 강도와 상관관계가 가장 큰 변수 중 하나였습니다.
`plot='correlation'`과 `feature='age'`를 함께 쓰면, 양생 기간이 길어질수록 SHAP 값이 어떻게 변하는지 확인할 수 있습니다.

In [ ]:
interpret_model(best_model, plot='correlation', feature='age')

## STEP 10. 성능 더 끌어올리기 - 하이퍼파라미터 튜닝

### 하이퍼파라미터란?

모델이 데이터로부터 "학습"하는 값(가중치 등)이 아니라, **학습을 시작하기 전에 사람이 미리 설정해야 하는 값**입니다.

예) LightGBM의 경우 - 트리의 개수, 트리의 최대 깊이, 학습률(learning rate) 등

### 📷 비유 - 카메라 설정

- **튜닝 전**: 새 카메라를 사서 기본(자동) 설정 그대로 사용
- **튜닝 후**: 촬영 환경(조명, 거리 등)에 맞게 조리개·셔터스피드를 최적으로 조정

같은 카메라(모델)인데, 설정을 맞추면 결과물(예측 성능)이 달라집니다.

### 💡 PyCaret의 해답 - `tune_model()`

`tune_model()` 한 줄로, 수십~수백 가지의 하이퍼파라미터 조합을 자동으로 시도해보고
**교차검증 성능이 가장 좋은 조합**을 찾아 적용한 모델을 반환합니다.

`n_iter`는 몇 가지 조합을 시도할지 정하는 값입니다 (값이 클수록 더 꼼꼼히 탐색하지만 시간이 더 걸립니다).

In [ ]:
tuned_model = tune_model(best_model, n_iter=30, optimize='R2')
tuned_results = pull()

tuned_results.tail(2)[['MAE', 'RMSE', 'R2']]

### 튜닝 전후 비교

아래에서 `best_model`(튜닝 전)과 `tuned_model`(튜닝 후)의 교차검증 평균 성능을 비교합니다.

### ⚠️ 튜닝이 항상 큰 차이를 만드는 건 아닙니다

이미 `compare_models()`에서 좋은 모델이 선택되었다면, 튜닝으로 얻는 추가 개선은 **크지 않을 수도 있습니다.**
(R²가 약 0.9215 → 0.925x 수준, RMSE가 약 4.64 → 4.5x MPa 수준으로 소폭 개선)

그래도 "추가로 시도해볼 만한 한 줄"이라는 점에서 실무적으로 의미가 있습니다.

In [ ]:
before = results_df[['MAE', 'RMSE', 'R2']].iloc[0]
after = tuned_results.loc['Mean', ['MAE', 'RMSE', 'R2']]

compare_df = pd.DataFrame({'튜닝 전': before, '튜닝 후': after})
compare_df

## STEP 11. (참고) 더 세밀한 튜닝이 필요하다면 - Optuna

`tune_model()`은 PyCaret이 탐색 범위와 방식을 알아서 정해줍니다. 편리하지만 제어할 수 없습니다.

**Optuna**는 같은 하이퍼파라미터 탐색을 **직접 설계**할 수 있는 라이브러리입니다.
- "learning_rate는 0.01~0.3 사이에서, n_estimators는 100~1000 사이에서 찾아라"처럼 탐색 범위를 직접 지정
- 시도할 횟수(`n_trials`)도 직접 설정
- 탐색 전략으로 베이지안 최적화를 기본으로 사용 → `tune_model()`보다 효율적으로 좋은 조합을 찾는 경우가 많음

아래는 Optuna로 LightGBM을 튜닝하는 최소한의 예시입니다. (실행하면 약 1~2분 소요)

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)  # 로그 최소화

X = df.drop(columns=['concrete_compressive_strength'])
y = df['concrete_compressive_strength']

def objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 3, 10),
        'num_leaves':    trial.suggest_int('num_leaves', 20, 150),
        'n_jobs': -1,
        'random_state': 123,
    }
    model = lgb.LGBMRegressor(**params)
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"최적 R²: {study.best_value:.4f}")
print(f"최적 파라미터: {study.best_params}")

### PyCaret tune_model() vs Optuna 비교

| | `tune_model()` | Optuna |
|---|---|---|
| 코드 난이도 | 한 줄 | 함수 직접 작성 필요 |
| 탐색 범위 | PyCaret이 자동 설정 | 직접 지정 |
| 탐색 전략 | Random Search | 베이지안 최적화 (기본) |
| 적합한 상황 | 빠르게 확인할 때 | 성능을 끝까지 짜낼 때 |

현업에서 "일단 돌려보기"는 `tune_model()`, "진지하게 최적화"는 Optuna를 선택하는 경우가 많습니다.

## 📝 오늘 한 일 정리

```
원본 데이터 (concrete_data.csv)
        ↓
   setup() → 전처리 자동화
        ↓
compare_models() → 수십 개 모델 한 번에 비교
        ↓
   결과 표 (R², RMSE, MAE)
        ↓
interpret_model() → SHAP Summary Plot (전체 변수 중요도)
                 → SHAP Force Plot   (샘플 1건 개별 설명)
        ↓
tune_model() → 하이퍼파라미터 자동 튜닝
        ↓
(참고) Optuna → 더 세밀한 튜닝이 필요할 때 ✅
```

### ✅ 핵심 포인트

- AutoML은 "어떤 모델이 좋을지"를 빠르게 비교해주는 도구
- SHAP Summary: 전체적으로 어떤 변수가 중요한가
- SHAP Force Plot: 특정 배합 하나가 왜 그 수치가 나왔는가
- 튜닝은 `tune_model()` 한 줄로 충분하지만, 더 세밀하게 하려면 Optuna

### 🔮 다음 세션 예고

지금까지는 모델이 Jupyter Notebook 안에서만 존재했습니다.
다음 세션에서는 이 모델을 **누구나 웹 브라우저에서 사용해볼 수 있는 간단한 앱(Gradio)**으로 만들어봅니다.